In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# =========================
# File paths
# =========================
INPUT_DIR = Path.cwd() / "output"

PAIR_FILE = INPUT_DIR / "generated_questions_pair.csv"
SINGLE_FILE = INPUT_DIR / "generated_questions_single.csv"
TRIPLE_FILE = INPUT_DIR / "generated_questions_triple.csv"

PAIR_OUT = INPUT_DIR / "generated_questions_pair_scored.csv"
SINGLE_OUT = INPUT_DIR / "generated_questions_single_scored.csv"
TRIPLE_OUT = INPUT_DIR / "generated_questions_triple_scored.csv"


# =========================
# Load embedding model
# =========================
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def compute_corr_score_within_group(group_df: pd.DataFrame) -> pd.DataFrame:
    """
    For one target_dims group:
    - Encode all texts into embeddings
    - Compute cosine similarity matrix
    - For each question, compute average cosine similarity with all other questions
    - Use this average as corr_score
    """
    group_df = group_df.copy().reset_index(drop=True)

    texts = group_df["generated_text"].astype(str).tolist()

    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    sim_matrix = cosine_similarity(embeddings)

    scores = []
    n = len(group_df)

    for i in range(n):
        others = np.delete(sim_matrix[i], i)
        score = float(others.mean()) if len(others) > 0 else 1.0
        scores.append(score)

    group_df["corr_score"] = scores
    return group_df


def process_one_file(input_path: Path, output_path: Path) -> pd.DataFrame:
    """
    Process one CSV file:
    - Keep only target_dims and generated_text
    - Compute corr_score within each target_dims group
    - Save output CSV with target_dims, generated_text, corr_score
    """
    print(f"Processing: {input_path}")

    df = pd.read_csv(input_path)

    # Keep only required columns
    df = df[["target_dims", "generated_text"]].copy()

    # Drop missing values
    df = df.dropna(subset=["target_dims", "generated_text"]).reset_index(drop=True)

    # Compute scores per dimension group
    result_parts = []
    for target_dims, group in df.groupby("target_dims", sort=False):
        scored_group = compute_corr_score_within_group(group)
        result_parts.append(scored_group)

    result = pd.concat(result_parts, ignore_index=True)

    # Sort within each dimension group by corr_score
    result = result.sort_values(
        by=["target_dims", "corr_score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # Keep only final columns
    result = result[["target_dims", "generated_text", "corr_score"]]

    # Save file
    result.to_csv(output_path, index=False)

    print(f"Saved to: {output_path}")
    print(f"Rows: {len(result)}")
    print()

    return result


# =========================
# Process all three files
# =========================
single_scored = process_one_file(SINGLE_FILE, SINGLE_OUT)
pair_scored = process_one_file(PAIR_FILE, PAIR_OUT)
triple_scored = process_one_file(TRIPLE_FILE, TRIPLE_OUT)

/Users/haikeyu/Desktop/mentalhealth-dimension-reduction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1667.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_single.csv
Saved to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_single_scored.csv
Rows: 240

Processing: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_pair.csv
Saved to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_pair_scored.csv
Rows: 840

Processing: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_triple.csv
Saved to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_triple_scored.csv
Rows: 1680

